# temporal_event_model v1 cache-v2 probe architecture

This notebook builds the actual v1 cache-v2 downstream model: one compact current event chunk goes through the real frozen v20 event encoder, then a simple nonlinear MLP decoder predicts the two future chunk labels used by the cache probe.

There is intentionally no `CONTEXT_CHUNKS` dimension here. Cache v2 stores one current `x` chunk per row, so the event encoder output is `[B, 32]`, and the probe output is `[B, 2, 5]` for two future chunks and five direction classes.


In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'research').exists()), Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ROOT


WindowsPath('D:/TradingCodes/quant-research-workbench')

In [2]:
import importlib
import sys

import torch
from torch import nn

from research.masked_event_model.v20.config import ModelConfig as V20ModelConfig
from research.masked_event_model.v20.model import EventTokenMaskedAutoencoder

# Jupyter kernels keep imported modules alive across cell edits. Reload the v1
# model module so the notebook sees newly added classes without requiring a
# full kernel restart.
MODEL_MODULE = "research.temporal_event_model.v1.model"
if MODEL_MODULE in sys.modules:
    temporal_v1_model = importlib.reload(sys.modules[MODEL_MODULE])
else:
    temporal_v1_model = importlib.import_module(MODEL_MODULE)
SingleChunkFutureLabelPredictor = temporal_v1_model.SingleChunkFutureLabelPredictor

CLASS_NAMES = ["strong_down", "down", "flat", "up", "strong_up"]

WORKSTATION_ROOT = Path(r"\\DESKTOP-SAAI85T\Workstation-D")
V20_RUN = WORKSTATION_ROOT / "TradingML" / "runtimes" / "masked_event_model" / "v20" / "pretrain" / "v20-fullpretrain-sharddecay-fixedmask070-emb32-bs8192-3epochs"
CHECKPOINTS = {
    "epoch1": V20_RUN / "checkpoints" / "checkpoint_step_000130176.pt",
    "epoch2": V20_RUN / "checkpoints" / "checkpoint_step_000260352.pt",
    "latest": V20_RUN / "checkpoints" / "checkpoint_latest.pt",
}

# Choose the pretrained event encoder checkpoint here.
ENCODER_CHECKPOINT_NAME = "epoch2"
ENCODER_CHECKPOINT = CHECKPOINTS[ENCODER_CHECKPOINT_NAME]

# Optional trained cache-probe checkpoint. Leave as None to inspect the architecture with a random MLP decoder.
PROBE_CHECKPOINT = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 1
TARGET_CHUNKS = 2
EVENTS_PER_CHUNK = 128
HEADER_BYTES = 14
EVENT_BYTES = 16
EMBEDDING_DIM = 32
HIDDEN_DIM = 128
DROPOUT = 0.10

print(f"root={ROOT}")
print(f"device={DEVICE}")
print(f"encoder_checkpoint={ENCODER_CHECKPOINT}")
print(f"probe_checkpoint={PROBE_CHECKPOINT if PROBE_CHECKPOINT else '<none>'}")
print(f"input: header=[B,{HEADER_BYTES}] events=[B,{EVENTS_PER_CHUNK},{EVENT_BYTES}]")
print(f"output: class_logits=[B,{TARGET_CHUNKS},{len(CLASS_NAMES)}] classes={CLASS_NAMES}")


root=D:\TradingCodes\quant-research-workbench
device=cuda
encoder_checkpoint=\\DESKTOP-SAAI85T\Workstation-D\TradingML\runtimes\masked_event_model\v20\pretrain\v20-fullpretrain-sharddecay-fixedmask070-emb32-bs8192-3epochs\checkpoints\checkpoint_step_000260352.pt
probe_checkpoint=<none>
input: header=[B,14] events=[B,128,16]
output: class_logits=[B,2,5] classes=['strong_down', 'down', 'flat', 'up', 'strong_up']


In [3]:
def load_v20_encoder(checkpoint: Path, device: torch.device) -> nn.Module:
    if not checkpoint.exists():
        raise FileNotFoundError(f"Encoder checkpoint not found: {checkpoint}")
    encoder_config = V20ModelConfig(
        input_representation="bit",
        d_byte=40,
        d_model=256,
        embedding_dim=EMBEDDING_DIM,
        n_heads=8,
        encoder_layers=10,
        decoder_layers=4,
        ffn_mult=4,
        dropout=0.08,
        decoder_force_fp32=False,
    )
    autoencoder = EventTokenMaskedAutoencoder(events_per_chunk=EVENTS_PER_CHUNK, config=encoder_config)
    payload = torch.load(checkpoint, map_location="cpu")
    if isinstance(payload, dict):
        state = payload.get("model_state_dict") or payload.get("model") or payload.get("state_dict")
    else:
        state = payload
    if not isinstance(state, dict):
        raise RuntimeError(f"Checkpoint does not contain a model state dict: {checkpoint}")
    missing, unexpected = autoencoder.load_state_dict(state, strict=False)
    if missing:
        print(f"encoder load missing keys: {len(missing)}")
    if unexpected:
        print(f"encoder load unexpected keys: {len(unexpected)}")
    encoder = autoencoder.build_encoder_model().to(device).eval()
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    return encoder


def build_probe_model(device: torch.device) -> SingleChunkFutureLabelPredictor:
    event_encoder = load_v20_encoder(ENCODER_CHECKPOINT, device)
    model = SingleChunkFutureLabelPredictor(
        event_encoder=event_encoder,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        target_chunks=TARGET_CHUNKS,
        classes=len(CLASS_NAMES),
        dropout=DROPOUT,
    ).to(device).eval()
    if PROBE_CHECKPOINT is not None and Path(PROBE_CHECKPOINT).is_file():
        payload = torch.load(PROBE_CHECKPOINT, map_location="cpu")
        if isinstance(payload, dict) and "decoder" in payload:
            missing, unexpected = model.decoder.load_state_dict(payload["decoder"], strict=False)
        else:
            state = payload.get("model") or payload.get("state_dict") if isinstance(payload, dict) else payload
            missing, unexpected = model.load_state_dict(state, strict=False)
        print(f"loaded probe checkpoint: {PROBE_CHECKPOINT}")
        print(f"probe load missing={len(missing)} unexpected={len(unexpected)}")
    return model

model = build_probe_model(DEVICE)
for p in model.event_encoder.parameters():
    assert not p.requires_grad

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
print(f"total_params={total_params:,}")
print(f"trainable_mlp_decoder_params={trainable_params:,}")
print(f"frozen_encoder_params={frozen_params:,}")


total_params=25,074,602
trainable_mlp_decoder_params=22,346
frozen_encoder_params=25,052,256


In [4]:
header_uint8 = torch.randint(0, 256, (BATCH_SIZE, HEADER_BYTES), dtype=torch.uint8, device=DEVICE)
events_uint8 = torch.randint(0, 256, (BATCH_SIZE, EVENTS_PER_CHUNK, EVENT_BYTES), dtype=torch.uint8, device=DEVICE)

with torch.no_grad():
    output = model(header_uint8, events_uint8)

print("chunk_embedding", tuple(output.chunk_embedding.shape))
print("class_logits", tuple(output.class_logits.shape))


chunk_embedding (1, 32)
class_logits (1, 2, 5)


In [5]:
class SingleChunkTensorSummaryModel(nn.Module):
    """Return plain tensors so torchinfo/torchview do not have to parse a dataclass."""

    def __init__(self, model: SingleChunkFutureLabelPredictor) -> None:
        super().__init__()
        self.event_encoder = model.event_encoder
        self.decoder = model.decoder

    def forward(self, header_uint8: torch.Tensor, events_uint8: torch.Tensor):
        chunk_embedding = self.event_encoder(header_uint8, events_uint8)
        class_logits = self.decoder(chunk_embedding.float())
        return class_logits, chunk_embedding


def print_ascii_safe(value) -> None:
    text = str(value).encode("ascii", "backslashreplace").decode("ascii")
    print(text)

summary_model = SingleChunkTensorSummaryModel(model).eval()
try:
    from torchinfo import summary
    summary_result = summary(
        summary_model,
        input_data=(header_uint8, events_uint8),
        depth=8,
        col_names=("input_size", "output_size", "num_params", "trainable"),
        verbose=0,
    )
    print_ascii_safe(summary_result)
except Exception as exc:
    print_ascii_safe(f"torchinfo summary failed: {exc!r}")


Layer (type:depth-idx)                             Input Shape               Output Shape              Param #                   Trainable
SingleChunkTensorSummaryModel                      [1, 14]                   [1, 2, 5]                 --                        Partial
\u251c\u2500EventChunkEncoder: 1-1                           [1, 14]                   [1, 32]                   --                        False
\u2502    \u2514\u2500VisibleEventTokenSelector: 2-1              [1, 128, 16]              [1, 128, 16]              --                        --
\u2502    \u2514\u2500HeaderTokenEncoder: 2-2                     [1, 14]                   [1, 1, 256]               --                        False
\u2502    \u2502    \u2514\u2500UInt8BytesToSignedBitFeatures: 3-1     [1, 14]                   [1, 112]                  --                        --
\u2502    \u2502    \u2514\u2500Sequential: 3-2                        [1, 112]                  [1, 256]                  --     

In [6]:
try:
    from torchview import draw_graph
    graph = draw_graph(
        summary_model,
        input_data=(header_uint8, events_uint8),
        expand_nested=True,
        graph_name="temporal_v1_single_chunk_cache_probe",
        save_graph=False,
    )
    graph.visual_graph
except Exception as exc:
    print(f"torchview graph failed: {exc!r}")


In [ ]:
with torch.no_grad():
    probs = torch.sigmoid(output.class_logits).detach().cpu()

print("probability range", float(probs.min()), float(probs.max()))
print("future chunk 0 class probs", dict(zip(CLASS_NAMES, probs[0, 0].tolist())))
print("future chunk 1 class probs", dict(zip(CLASS_NAMES, probs[0, 1].tolist())))


probability range 0.2844390869140625 0.684049665927887
future chunk 0 class probs {'strong_down': 0.5444769859313965, 'down': 0.2968423664569855, 'flat': 0.684049665927887, 'up': 0.6757077574729919, 'strong_up': 0.4801008403301239}
future chunk 1 class probs {'strong_down': 0.5446686744689941, 'down': 0.2844390869140625, 'flat': 0.3534385859966278, 'up': 0.4645005166530609, 'strong_up': 0.5872227549552917}


: 